# Denoising Diffusion Probabilistic Model (DDPM)

Trains a U-Net to predict noise on 32×32 pixel-art images using the DDPM framework.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import math
import os

In [ ]:
# ── Noise schedule (linear beta schedule from DDPM paper) ──────────────────
def get_beta_schedule(timesteps=1000):
    beta_start = 0.0001
    beta_end   = 0.02
    return np.linspace(beta_start, beta_end, timesteps, dtype=np.float32)

TIMESTEPS = 1000
betas          = get_beta_schedule(TIMESTEPS)
alphas         = 1.0 - betas
alphas_cumprod = np.cumprod(alphas)           # ᾱ_t used in DDPM formula

# Pre-cast to TF tensors for fast GPU lookup
betas_tf          = tf.constant(betas,          dtype=tf.float32)
alphas_tf         = tf.constant(alphas,         dtype=tf.float32)
alphas_cumprod_tf = tf.constant(alphas_cumprod, dtype=tf.float32)


# ── Forward diffusion ───────────────────────────────────────────────────────
def add_noise(img, t):
    """
    img : clean image tensor, values in [0, 1]
    t   : integer timestep index (0 … TIMESTEPS-1)
    Returns: (noisy_img, noise)
    """
    alpha_bar_t           = tf.gather(alphas_cumprod_tf, t)
    noise                 = tf.random.normal(tf.shape(img))
    sqrt_alpha_bar        = tf.sqrt(alpha_bar_t)
    sqrt_one_minus_alpha_bar = tf.sqrt(1.0 - alpha_bar_t)
    noisy_img = sqrt_alpha_bar * img + sqrt_one_minus_alpha_bar * noise
    return noisy_img, noise


print(f"✅ Noise schedule ready  (T={TIMESTEPS}, β_start={betas[0]:.4f}, β_end={betas[-1]:.4f})")

In [ ]:
# ── Optional: visualise a noisy image at different timesteps ──────────────
# Replace the path below with any JPEG on your system, or skip this cell.
IMAGE_PATH = "./DSC_7047.jpg"   # ← change if needed

if os.path.exists(IMAGE_PATH):
    raw   = tf.io.read_file(IMAGE_PATH)
    image = tf.image.decode_jpeg(raw, channels=3)
    image = tf.image.resize_with_crop_or_pad(image, 32, 32)
    image = tf.cast(image, tf.float32) / 255.0   # normalise to [0, 1]

    fig, axes = plt.subplots(1, 5, figsize=(14, 3))
    steps = [0, 250, 500, 750, 999]
    for ax, t in zip(axes, steps):
        noisy, _ = add_noise(image, t)
        ax.imshow(tf.clip_by_value(noisy, 0, 1).numpy())
        ax.set_title(f"t={t}")
        ax.axis("off")
    plt.suptitle("Forward diffusion at different timesteps")
    plt.tight_layout()
    plt.show()
else:
    print(f"⚠️  '{IMAGE_PATH}' not found — skipping visualisation.")

In [ ]:
from tensorflow.keras.layers import (
    BatchNormalization, Input, Conv2D, Conv2DTranspose,
    Layer, Dense, Reshape, Add
)
from tensorflow.keras import Model


class TimestepEmbedding(Layer):
    """Sinusoidal positional encoding for the diffusion timestep."""
    def __init__(self, dim=128, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim

    def call(self, timesteps):
        half_dim = self.dim // 2
        emb = math.log(10000.0) / (half_dim - 1)
        emb = tf.exp(tf.range(half_dim, dtype=tf.float32) * -emb)
        emb = tf.cast(timesteps, tf.float32) * emb[None, :]
        emb = tf.concat([tf.sin(emb), tf.cos(emb)], axis=-1)
        return emb


def inject_timestep(x, t_emb, filters, name_suffix):
    """Projects timestep embedding and broadcasts it over spatial dims."""
    t_proj = Dense(filters, name=f"time_proj_{filters}_{name_suffix}")(t_emb)
    t_proj = Reshape((1, 1, filters))(t_proj)
    return Add()([x, t_proj])


def build_unet():
    """Builds a U-Net that predicts the noise added to an image at timestep t."""
    input_image    = Input(shape=(32, 32, 3), name="image")
    input_timestep = Input(shape=(1,),        name="timestep")

    t_emb = TimestepEmbedding(dim=128)(input_timestep)

    # ── Encoder ──────────────────────────────────────────────────────────────
    x = Conv2D(32,  (3, 3), padding="same", activation="relu")(input_image)
    x = BatchNormalization()(x)
    x = inject_timestep(x, t_emb, 32, "enc1")
    enc1 = x                                           # skip connection

    x = Conv2D(64,  (5, 5), strides=(2, 2), padding="same", activation="relu")(x)
    x = BatchNormalization()(x)
    x = inject_timestep(x, t_emb, 64, "enc2")
    enc2 = x                                           # skip connection

    x = Conv2D(128, (3, 3), strides=(2, 2), padding="same", activation="relu")(x)
    x = BatchNormalization()(x)
    x = inject_timestep(x, t_emb, 128, "enc3")

    # ── Decoder ──────────────────────────────────────────────────────────────
    x = Conv2DTranspose(64, (3, 3), strides=(2, 2), padding="same", activation="relu")(x)
    x = BatchNormalization()(x)
    x = inject_timestep(x, t_emb, 64, "dec1")
    x = Add()([x, enc2])                               # skip

    x = Conv2DTranspose(32, (5, 5), strides=(2, 2), padding="same", activation="relu")(x)
    x = BatchNormalization()(x)
    x = inject_timestep(x, t_emb, 32, "dec2")
    x = Add()([x, enc1])                               # skip

    x = Conv2DTranspose(32, (3, 3), padding="same", activation="relu")(x)
    x = BatchNormalization()(x)
    x = inject_timestep(x, t_emb, 32, "dec3")

    # Output: predicted noise — NO activation (values can be negative)
    outputs = Conv2D(3, (3, 3), padding="same")(x)

    model = Model(inputs=[input_image, input_timestep], outputs=outputs)
    return model


model = build_unet()
print("✅ Model built successfully!")
model.summary()

# Quick sanity-check forward pass
test_img = np.random.randn(2, 32, 32, 3).astype(np.float32)
test_t   = np.array([[350], [720]], dtype=np.float32)
out      = model([test_img, test_t])
print(f"\n✅ Forward pass OK — output shape: {out.shape}")

In [ ]:
import kagglehub

path = kagglehub.dataset_download("ebrahimelgazar/pixel-art")
print("Dataset path:", path)

In [ ]:
# ── tf.data pipeline ────────────────────────────────────────────────────────
IMAGE_DIR  = os.path.join(path, "images", "images")
BATCH_SIZE = 32

def preprocess(filename):
    """Load → resize → normalise → random timestep → add DDPM noise."""
    image = tf.io.read_file(IMAGE_DIR + "/" + filename)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize_with_crop_or_pad(image, 32, 32)
    image = tf.cast(image, tf.float32) / 255.0          # [0, 1]

    t            = tf.random.uniform((), minval=0, maxval=TIMESTEPS, dtype=tf.int32)
    noisy_img, noise = add_noise(image, t)

    # inputs → (noisy_image, timestep);  target → noise
    return (noisy_img, tf.cast(tf.reshape(t, (1,)), tf.float32)), noise


filenames = tf.constant(os.listdir(IMAGE_DIR))
tf_ds = (
    tf.data.Dataset.from_tensor_slices(filenames)
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print(f"✅ Dataset ready — {len(filenames)} images, batch size {BATCH_SIZE}")

In [ ]:
EPOCHS     = 10          # increase for better quality (50–200 recommended)
CKPT_PATH  = "./ckpt_best.weights.h5"

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    # FIX: 'accuracy' was removed — meaningless for noise regression.
    # Only MSE (the DDPM objective) is tracked.
    loss=tf.keras.losses.MeanSquaredError(),
)

callbacks = [
    # Save the weights whenever val_loss improves
    tf.keras.callbacks.ModelCheckpoint(
        filepath=CKPT_PATH,
        save_best_only=True,
        save_weights_only=True,
        monitor="loss",
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
]

history = model.fit(tf_ds, epochs=EPOCHS, callbacks=callbacks)

# Persist the full model so it can be reloaded later
model.save("./diffusion_model.keras")
print("\n✅ Model saved to ./diffusion_model.keras")

# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="MSE loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss (DDPM noise prediction)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── DDPM Reverse Diffusion (ancestral sampling) ──────────────────────────────
# FIX: loop now runs from T-1 = 999 down to 1 (was incorrectly cut at 300).

def ddpm_sample(model, num_samples=1, seed=None):
    """
    Generate images by running the full DDPM reverse process.
    Returns a (num_samples, 32, 32, 3) tensor with values in [0, 1].
    """
    if seed is not None:
        tf.random.set_seed(seed)

    # Start from pure Gaussian noise (x_T)
    x = tf.random.normal((num_samples, 32, 32, 3))

    for t in reversed(range(1, TIMESTEPS)):          # FIX: full range 999 → 1
        t_batch   = tf.constant([[float(t)]] * num_samples)  # (N, 1)
        pred_noise = model([x, t_batch], training=False)

        alpha_t     = alphas[t]
        alpha_bar_t = alphas_cumprod[t]
        beta_t      = betas[t]

        # DDPM reverse step  (Eq. 11 from Ho et al. 2020)
        x = (
            (1.0 / np.sqrt(alpha_t)) *
            (x - (beta_t / np.sqrt(1.0 - alpha_bar_t)) * pred_noise)
        )

        if t > 1:
            x = x + np.sqrt(beta_t) * tf.random.normal(tf.shape(x))

    return tf.clip_by_value(x, 0.0, 1.0)


# ── Generate and display a grid of samples ───────────────────────────────────
NUM_SAMPLES = 8
samples = ddpm_sample(model, num_samples=NUM_SAMPLES, seed=42)

fig, axes = plt.subplots(1, NUM_SAMPLES, figsize=(2 * NUM_SAMPLES, 2))
for ax, img in zip(axes, samples):
    ax.imshow(img.numpy())
    ax.axis("off")
plt.suptitle(f"Generated pixel-art samples ({NUM_SAMPLES}×)")
plt.tight_layout()
plt.show()

print("✅ Sampling complete")